# KAF-2 — Consumer lag & offset semantics

**Break → Detect → Fix → Prove.** Producers append to a topic; consumers read and **commit** how
far they've gotten. The gap between them — **lag** = `end offset − committed offset` — is the
headline health metric for any consumer. And *how* you commit (auto vs manual, before vs after
processing) decides what happens to a message when a consumer crashes: **lose it**, or **reprocess** it.

**Pre-requisite:** `make up` (Kafka at `localhost:29092` for producers/consumers, `kafka:9092` for
Spark). Inspect consumer lag live in **kafka-ui** at http://localhost:8080 → topic → **Consumers**.
**Laptop-safe:** a bounded 2,000-message batch, no infinite loops; the topic is deleted at teardown.

See the companion [`README.md`](./README.md) and the [troubleshooting sheet](../../docs/troubleshooting.md).

In [ ]:
from common.spark_session import spark, display_df
from common.kafka_helpers import (ensure_topic, produce_events, topic_end_offsets,
                                  consumer_group_lag, delete_topic, BOOTSTRAP)
from kafka import KafkaConsumer
from pyspark.sql import functions as F

TOPIC    = "kaf2_orders"
GROUP    = "kaf2-app"
N        = 2000          # bounded backlog
PARTIAL  = 500           # how many the slow consumer gets through
print("producers/consumers ->", BOOTSTRAP, "| kafka-ui: http://localhost:8080")

## Step 0 — a topic with a full backlog

One partition keeps the lag arithmetic obvious. We produce a burst of `N = 2000` events; nothing
consumes them yet, so the whole batch is backlog.

In [ ]:
ensure_topic(TOPIC, num_partitions=1, recreate=True)
produce_events(TOPIC, N, value_fn=lambda i: {"order_id": i, "amount": i * 1.0})
print("end offsets (producer high-water mark):", topic_end_offsets(TOPIC))

## 1. Break it — a slow consumer falls behind

A realistic under-provisioned consumer: it processes messages slowly and only gets through **part**
of the backlog before we check on it. We read ~`PARTIAL` messages, **commit**, then stop. The
committed offset now trails the end offset — that gap is **lag**.

We disable auto-commit here so the commit is explicit and deterministic (`commit()` after the read).

In [ ]:
def consume_n(group_id, n, *, enable_auto_commit, commit_at_end):
    """Read up to n messages with a fresh consumer, then close (simulating a stop/crash).
    Returns the order_ids seen. If commit_at_end, manually commit AFTER 'processing'."""
    c = KafkaConsumer(TOPIC, bootstrap_servers=BOOTSTRAP, group_id=group_id,
                      enable_auto_commit=enable_auto_commit, auto_offset_reset="earliest",
                      consumer_timeout_ms=4000, value_deserializer=lambda b: b)
    seen = []
    try:
        for msg in c:
            seen.append(msg.offset)
            if len(seen) >= n:
                break
        if commit_at_end:
            c.commit()           # commit only after the batch is 'processed'
    finally:
        c.close()
    return seen

seen = consume_n(GROUP, PARTIAL, enable_auto_commit=False, commit_at_end=True)
print(f"slow consumer processed {len(seen)} messages, then stopped.")

## 2. Detect it — consumer-group lag

`consumer_group_lag(group, topic)` returns per-partition `{committed, end, lag}`. The same number
shows live in **kafka-ui → topic `kaf2_orders` → Consumers → `kaf2-app`**. Lag ≈ `N − PARTIAL`:
the group is ~1,500 messages behind.

In [ ]:
lag_broken = consumer_group_lag(GROUP, TOPIC)
print("lag per partition:", lag_broken)
print("TOTAL LAG       :", sum(p["lag"] for p in lag_broken.values()), "  <- the group is this far behind")

## 3. Diagnose — offset commit semantics (the crash test)

Lag tells you *how far behind*. The **commit strategy** tells you what a **crash** does to in-flight
messages. The committed offset is a promise — "everything below this, I'm done." When you make that
promise matters:

- **Auto-commit** (`enable_auto_commit=True`): offsets commit on a timer, possibly **before** you
  finish processing → a crash **loses** those messages (at-most-once).
- **Manual commit after processing**: a crash **before** the commit means the next consumer
  **re-reads** them → **duplicates / reprocess** (at-least-once).

Let's demonstrate the at-least-once (reprocess) case directly: a consumer reads a batch but
**does not commit**, then "crashes" (closes). A **new** consumer in the **same group** resumes from
the last *commit* — so it re-reads the uncommitted messages.

In [ ]:
DUP_GROUP = "kaf2-dup"
ensure_topic(TOPIC, num_partitions=1, recreate=True)      # fresh backlog for a clean demo
produce_events(TOPIC, N, value_fn=lambda i: {"order_id": i, "amount": i * 1.0})

# Consumer #1: read 300, but CRASH before committing (commit_at_end=False, auto-commit off).
crashed = consume_n(DUP_GROUP, 300, enable_auto_commit=False, commit_at_end=False)
print(f"consumer #1 read {len(crashed)} msgs (offsets {crashed[0]}..{crashed[-1]}) then crashed WITHOUT committing")

# Consumer #2: same group, starts again -> re-reads the uncommitted offsets.
reread = consume_n(DUP_GROUP, 300, enable_auto_commit=False, commit_at_end=True)
overlap = sorted(set(crashed) & set(reread))
print(f"consumer #2 re-read {len(reread)} msgs; {len(overlap)} were ALREADY seen by #1 -> duplicates/reprocess")

## 4. Fix it — commit after processing + an idempotent sink, and drain the backlog

Two things:
1. **Commit after processing** (what `consume_n(..., commit_at_end=True)` does) → at-least-once: a
   crash reprocesses instead of losing. Pair with an **idempotent sink** (dedupe by `order_id` /
   offset when writing — e.g. `MERGE` into Iceberg) so reprocessing is harmless = effective
   exactly-once end-to-end (continued in **STR-2**).
2. **Right-size consumers/partitions** so you drain at least as fast as you fill, keeping lag ≈ 0.

Here we simply **drain the rest of the backlog** for `kaf2-app` (read & commit everything left).

In [ ]:
# Drain the remainder for the original group (commit after processing).
drained = consume_n(GROUP, N, enable_auto_commit=False, commit_at_end=True)
print(f"drained {len(drained)} remaining messages and committed.")

# Idempotent-sink intuition: even with the duplicate reads above, dedup by order_id yields N unique rows.
all_offsets = sorted(set(crashed) | set(reread))   # union across the crash + reprocess
print(f"duplicate-tolerant sink: {len(crashed)+len(reread)} reads -> {len(all_offsets)} unique offsets after dedup")

## 5. Prove it — lag before (high) → after (≈0)

The same `consumer_group_lag` helper, before vs after draining `kaf2-app`. Committed catches up to
end; lag collapses to ~0.

In [ ]:
lag_fixed = consumer_group_lag(GROUP, TOPIC)
def total(d): return sum(p["lag"] for p in d.values())
print(f"{'state':<28}{'committed':>10}{'end':>8}{'lag':>8}")
for name, d in [("partial read (broken)", lag_broken), ("after draining (fixed)", lag_fixed)]:
    c = sum(p["committed"] for p in d.values()); e = sum(p["end"] for p in d.values())
    print(f"{name:<28}{c:>10}{e:>8}{total(d):>8}")
print(f"\nLAG: {total(lag_broken)} -> {total(lag_fixed)}   (caught up = healthy)")

## Takeaways & "in real production…"

- **Alert on consumer lag.** Rising lag is the earliest sign consumers can't keep up; flat-zero is
  healthy. (Per-partition lag also reveals hot-partition skew — see **KAF-1**.)
- **Choose auto vs manual commit deliberately.** Auto-commit risks **loss** on crash; manual commit
  *after* processing gives **at-least-once**.
- **At-least-once + idempotent sink = effective exactly-once.** Make reprocessing harmless rather
  than chasing perfect once-only delivery (continued in **STR-2**).
- **Right-size for throughput** — enough partitions/consumers to drain as fast as you fill, so lag
  never grows into an outage.

## Teardown

In [ ]:
delete_topic(TOPIC)
print("Deleted", TOPIC, "(`make clean` also clears any local state).")